<p style="text-align: center">
<img src="../../../assets/images/dtlogo.png" alt="Duckietown" width="50%">
</p>


# Object Detection - Dataset Setup

In this notebook you will:

1. Download the raw labeled dataset.
2. Implement a function that converts labelme annotations to YOLO format.
3. Run the conversion script to produce a training-ready dataset.

The next notebook ([Training](../03-Training/training.ipynb)) trains the model on Google Colab.


## 1. Setup


In [1]:
import sys
import os

notebook_dir = os.path.abspath('')
project_root = os.path.abspath(os.path.join(notebook_dir, '..', '..', '..', '..'))
packages_dir = os.path.join(project_root, 'tasks', 'object_detection', 'packages')

if project_root not in sys.path:
    sys.path.insert(0, project_root)
if packages_dir not in sys.path:
    sys.path.insert(0, packages_dir)


In [2]:
%load_ext autoreload
%autoreload 2


## 2. Download the raw dataset

The cell below downloads the raw labeled images and their labelme JSON annotations from HuggingFace.
This is the starting point -- images are not yet in YOLO format.


In [ ]:
import urllib.request
import zipfile

RAW_DIR = os.path.join(project_root, "tasks", "object_detection", "dataset", "raw")
zip_path = os.path.join(project_root, "tasks", "object_detection", "dataset", "raw_dataset.zip")

if os.path.exists(RAW_DIR) and os.listdir(RAW_DIR):
    print("Raw dataset already exists, skipping download.")
else:
    os.makedirs(os.path.dirname(zip_path), exist_ok=True)
    print("Downloading raw dataset from HuggingFace...")
    url = "https://huggingface.co/datasets/giokezo/duckietown_raw_dataset/resolve/main/raw_dataset.zip"
    urllib.request.urlretrieve(url, zip_path)
    print("Extracting...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(os.path.join(project_root, "tasks", "object_detection", "dataset"))
    os.remove(zip_path)
    print(f"Done. Raw images and labels are in: {RAW_DIR}")


## 3. Your task - implement `convert_labelme_json`

Open [`tasks/object_detection/packages/dataset_activity.py`](../../packages/dataset_activity.py) 
and implement the `convert_labelme_json` function.

### What the function does

It takes a labelme JSON annotation file and converts its bounding boxes into YOLO format.
For each annotated object it should return one string:

```
<class_id> <cx> <cy> <w> <h>
```

where `cx, cy, w, h` are all **normalized to [0, 1]** relative to `IMAGE_SIZE x IMAGE_SIZE`.

### labelme JSON structure

Each annotation file looks like this:

```json
{
  "shapes": [
    {
      "label": "duckie",
      "shape_type": "rectangle",
      "points": [[120.0, 85.0], [210.0, 160.0]]
    }
  ]
}
```

`points` gives the **top-left and bottom-right corners** in the *original* image pixel space.

### Function signature

```python
def convert_labelme_json(json_path: str, img_w: int, img_h: int) -> List[str]:
```

- `json_path` — path to the `.json` file
- `img_w`, `img_h` — original image dimensions (before resizing)

### Steps

1. Open `json_path` and load it with `json.load()`.
2. Iterate over `data["shapes"]`. Skip any shape whose `label` is not in `CLASSES`.
3. Use `CLASSES.index(label)` to get the class ID.
4. Extract the two corner points and compute `xmin, ymin, xmax, ymax`.
5. Scale the corners from original image space to `IMAGE_SIZE` space:
   ```python
   xmin = xmin * IMAGE_SIZE / img_w
   xmax = xmax * IMAGE_SIZE / img_w
   ymin = ymin * IMAGE_SIZE / img_h
   ymax = ymax * IMAGE_SIZE / img_h
   ```
6. Compute normalized center and size:
   ```python
   cx = (xmin + xmax) / 2 / IMAGE_SIZE
   cy = (ymin + ymax) / 2 / IMAGE_SIZE
   w  = (xmax - xmin) / IMAGE_SIZE
   h  = (ymax - ymin) / IMAGE_SIZE
   ```
7. Append `f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"` to your list and return it.


## 4. Run the conversion

Once you have implemented `convert_labelme_json`, run the cell below.
It converts all raw images and annotations into a YOLO-ready dataset split into `train/` and `val/`.


In [ ]:
import subprocess
script = os.path.join(project_root, "tasks", "object_detection", "packages", "prepare_dataset.py")
result = subprocess.run(["python", script], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("Error:", result.stderr)


## 5. Next step

Your dataset is now in `tasks/object_detection/dataset/train/` and `val/`, ready for training.

Continue with the [Training notebook](../03-Training/training.ipynb) to zip it up and train on Google Colab.
